- Tensor Parallelism:
    - "Let's split the work inside a layer across GPUs."

- Pipeline Parallelism:
    - "Let's split the layers of the model across GPUs."

- Data Parallelism:
    - "Let's give different GPUs/groups different batches of data."

- GPU ↔ GPU communication = communication between accelerator devices.
- Intra-node communication = GPUs communicating within the same physical machine.
- Inter-node communication = GPUs communicating across physical machines.
- Layer-to-layer communication = usually the activation tensor being passed from one layer/stage to the next. In pipeline parallelism, this becomes GPU-to-GPU communication when layers are on different GPUs.
- Gradient communication = GPUs exchanging gradients during distributed training.

## Tensor Parallelism

- lots of GPU ↔ GPU communication
-              Same Transformer Layer
             
                 Input X
                /       \
               /         \
           GPU 0         GPU 1
            W1             W2
             │              │
             ▼              ▼
            Y1             Y2
             │              │
             └──────┬───────┘
                    ▼
                 Combine

- Depending on how the matrix is partitioned, this communication is typically implemented using collectives such as:
  - All-Gather
  - All-Reduce
  - Reduce-Scatter

- GPU ↔ GPU communication within the same computation/layer.

- TP communication can be done inter-node but **NOT** recommended as this is expensive compared with intra-node NVLink communication.
- So a common strategy is to keep the tensor-parallel group inside one node whenever possible.

## Pipeline Parallelism — activation communication between stages

- GPU 0                 GPU 1                 GPU 2

Layers 1–8            Layers 9–16           Layers 17–24
    │                     │                      │
    └──── activation ────►│
                          └──── activation ─────►

- GPU 0 finishes its layers and sends the resulting activation tensor to GPU 1.
- GPU 1 processes it and sends another activation tensor to GPU 2.
- So PP communication is fundamentally:
    - Activation communication between pipeline stages.


| Parallelism              | What is split?                 | Main communication                     | Typical collective                       |
| ------------------------ | ------------------------------ | -------------------------------------- | ---------------------------------------- |
| **Tensor Parallelism**   | Tensor/matrix within a layer   | Partial activations/results            | All-Reduce, All-Gather, Reduce-Scatter   |
| **Pipeline Parallelism** | Layers/stages                  | Activations + gradients between stages | Point-to-point send/receive              |
| **Data Parallelism**     | Training data / model replicas | Gradients                              | All-Reduce / Reduce-Scatter + All-Gather |
| **Sequence Parallelism** | Sequence dimension             | Activations/gradients                  | All-Gather / Reduce-Scatter              |


- TP → collective communication of partial tensors within a layer.
- PP → point-to-point activation/gradient communication between pipeline stages.
- DP → gradient synchronization between model replicas.
- Intra-node/inter-node → where those GPUs physically reside and therefore how expensive the communication is.

| Parallelism | Main Goal | Latency | Throughput | How it helps |
|---|---|---:|---:|---|
| **Tensor Parallelism (TP)** | Speed up computation of a single layer | ✅ Can reduce | ✅ Can increase | Splits the computation of each layer across multiple GPUs, so large matrix multiplications are performed in parallel |
| **Pipeline Parallelism (PP)** | Fit a large model across GPUs; improve utilization | ⚠️ Can reduce for large models, but not its primary benefit | ✅ Strong benefit with microbatches | Splits layers across GPUs and processes multiple microbatches simultaneously like a pipeline |
| **Data Parallelism (DP)** | Process more independent inputs | ❌ Usually does not reduce single-request latency | ✅ Strong benefit | Replicates the model across GPUs and gives each replica different data/requests |
| **Sequence Parallelism (SP)** | Reduce memory / improve efficiency | ⚠️ Sometimes | ✅ Can improve | Splits computation along the sequence dimension, usually together with TP |
| **Expert Parallelism (EP)** | Scale Mixture-of-Experts (MoE) models | ⚠️ Depends | ✅ Can greatly improve | Different experts are placed on different GPUs; tokens are routed only to selected experts |

| Requirement | First parallelism to think about |
|---|---|
| Make one large model fit across GPUs | **Pipeline Parallelism** |
| Make one layer's computation faster | **Tensor Parallelism** |
| Process more requests / samples per second | **Data Parallelism** |
| Train an MoE model efficiently | **Expert Parallelism** |
| Reduce activation memory while using TP | **Sequence Parallelism** |

- Tensor Parallelism → split computation within a layer → mainly GPU-to-GPU tensor communication.

- Pipeline Parallelism → split layers across GPUs → activation/gradient communication between stages.

- Data Parallelism → replicate the model → different data on each GPU → gradient synchronization.

- Intra-node → GPUs communicating within one physical machine.

- Inter-node → GPUs communicating across physical machines.

LLM Parallelism — Short Interview Notes
1. Latency vs Throughput
Latency: Time taken for a single request/work unit to complete.
Throughput: Amount of work completed per unit time, e.g. requests/sec or tokens/sec.
Training: Primarily a throughput problem — maximize tokens processed per second.
Inference: Often latency-sensitive, but at large scale it is also a throughput problem.
2. GPU vs Node
GPU: Accelerator that performs the tensor/matrix computations.
Node: A physical machine containing one or more GPUs.
Intra-node: Communication between GPUs within the same machine.
Inter-node: Communication between GPUs on different machines.
Node 0
├── GPU 0
├── GPU 1
├── GPU 2
└── GPU 3

Node 1
├── GPU 4
├── GPU 5
├── GPU 6
└── GPU 7

3. Main Parallelism Types
Tensor Parallelism — TP

Split tensors/matrices within a layer across GPUs.

             Transformer Layer
              /            \
           GPU 0          GPU 1
          W1 portion      W2 portion
              \            /
               → Combine ←

Multiple GPUs cooperate on the same layer.
Communication: partial tensors/activations.
Common collectives: All-Reduce, All-Gather, Reduce-Scatter.
Can reduce latency by parallelizing computation.
Can also improve throughput.
Communication-heavy → preferably keep TP GPUs within the same node/high-bandwidth interconnect.
Pipeline Parallelism — PP

Split layers across GPUs.

GPU 0 → Layers 1–8
GPU 1 → Layers 9–16
GPU 2 → Layers 17–24
GPU 3 → Layers 25–32

Each GPU owns a different group of layers.
Communication: activations between pipeline stages during forward pass and gradients during backward pass.
Uses microbatches to keep GPUs busy.
Main benefit: allows very large models to fit across GPUs and improves utilization.
Important overhead: pipeline bubble.
Data Parallelism — DP

Replicate the model and split the data.

             Model
            /     \
        GPU 0     GPU 1
       Batch A   Batch B

Every GPU has a copy of the model.
Different GPUs process different batches/requests.
Gradients are synchronized.
Main communication: gradient synchronization, commonly All-Reduce.
Excellent for increasing throughput.
Doesn't normally reduce the latency of one individual request.
Sequence Parallelism — SP
Splits computation/activations along the sequence dimension.
Usually used together with Tensor Parallelism.
Primarily helps memory efficiency and computational efficiency.
Expert Parallelism — EP

Used primarily for Mixture-of-Experts (MoE) models.

Tokens
  │
  ├── Expert 1 → GPU 0
  ├── Expert 2 → GPU 1
  ├── Expert 3 → GPU 2
  └── Expert 4 → GPU 3

Different experts are placed on different GPUs.
Tokens are routed to selected experts.
Allows very large MoE models to scale efficiently.
4. Latency vs Throughput Cheat Sheet
| Parallelism | What is Split? | Main Communication | Latency | Throughput | Primary Benefit |
|---|---|---|---|---|---|
| Tensor Parallelism (TP) | Tensors/matrices within a layer | Partial tensors/results | ✅ Can improve | ✅ Can improve | Faster computation of a layer |
| Pipeline Parallelism (PP) | Layers/stages | Activations + gradients | ⚠️ Not primary | ✅ Can improve | Fit large models + utilize GPUs |
| Data Parallelism (DP) | Data / model replicas | Gradients | ❌ Usually no | ✅ Strong improvement | Process more data/requests |
| Sequence Parallelism (SP) | Sequence dimension | Activations/gradients | ⚠️ Sometimes | ✅ Can improve | Memory + computation efficiency |
| Expert Parallelism (EP) | MoE experts | Token routing | ⚠️ Depends | ✅ Can improve significantly | Efficient MoE scaling |

Easy mental model
TP → "Make ONE layer work faster."

PP → "Split the MODEL across GPUs."

DP → "Process MORE data simultaneously."

SP → "Split the SEQUENCE."

EP → "Split the EXPERTS."

Interview Questions & Answers
🟢 Easy — 1-liners
1. What is latency?

Answer: Latency is the time required to complete a single unit of work, such as one LLM request.

2. What is throughput?

Answer: Throughput is the amount of work completed per unit of time, such as requests/sec or tokens/sec.

3. What is Tensor Parallelism?

Answer: Tensor Parallelism splits tensors or matrix computations within a layer across multiple GPUs.

4. What is Pipeline Parallelism?

Answer: Pipeline Parallelism splits different layers of a model across different GPUs.

5. What is Data Parallelism?

Answer: Data Parallelism replicates the model across GPUs and gives each replica different data.

6. What is a node?

Answer: A node is a physical machine that can contain one or more GPUs.

7. What is intra-node communication?

Answer: Communication between GPUs located within the same physical machine.

8. What is inter-node communication?

Answer: Communication between GPUs located on different physical machines.

🟡 Medium — 2-liners
9. How does Tensor Parallelism improve performance?

Answer: It divides a large tensor/matrix operation across multiple GPUs so they perform portions of the computation simultaneously. The resulting partial tensors are then combined through GPU-to-GPU communication.

10. How does Data Parallelism improve throughput?

Answer: Each GPU maintains a model replica and processes a different batch of data simultaneously. Therefore, more samples/tokens can be processed per unit time.

11. What communication occurs in Pipeline Parallelism?

Answer: Pipeline stages exchange activation tensors during the forward pass and gradients during the backward pass. This is generally point-to-point communication between neighboring stages.

12. Why is Tensor Parallelism communication-heavy?

Answer: Multiple GPUs must frequently exchange partial results while computing the same layer. Therefore, high-bandwidth GPU interconnects are particularly important for TP.

13. Why doesn't Data Parallelism usually reduce single-request latency?

Answer: DP gives different data to different model replicas rather than splitting the computation of one request across GPUs. Its main purpose is increasing the number of requests/samples processed concurrently.

14. What is the pipeline bubble?

Answer: The pipeline bubble is the period when some pipeline stages are idle while the pipeline is being filled or drained. Using more microbatches helps reduce this idle time.

15. Can Tensor Parallelism be inter-node?

Answer: Yes, TP can span multiple nodes, but this introduces network communication and is generally more expensive than keeping the TP group within one node.

🔴 Tougher — Detailed Answers
16. Compare Tensor Parallelism and Pipeline Parallelism.

Answer:

Tensor Parallelism splits the computation within a layer across multiple GPUs:

Layer
 ├── GPU 0 → portion of computation
 └── GPU 1 → portion of computation


Pipeline Parallelism instead splits the layers themselves:

GPU 0 → Layers 1–8
GPU 1 → Layers 9–16
GPU 2 → Layers 17–24


Therefore, TP requires frequent communication between GPUs working on the same layer, while PP primarily communicates activations between pipeline stages.

17. How would you choose between TP, PP and DP?

Answer:

It depends on the bottleneck:

Model doesn't fit on one GPU
        ↓
      TP / PP

Need faster computation of a large layer
        ↓
       TP

Need higher training/serving throughput
        ↓
       DP

Very large model + many GPUs
        ↓
     TP + PP + DP


TP is useful for splitting computation within layers, PP distributes model layers across devices, and DP creates multiple model replicas to process different data concurrently.

18. Why is TP usually kept within a node?

Answer:

TP involves frequent synchronization and exchange of partial tensors. If those GPUs are on different nodes, communication must travel through the network, which generally has higher latency and lower effective bandwidth than high-speed intra-node GPU interconnects.

Therefore, a common strategy is:

Node 0
GPU 0 ─ GPU 1 ─ GPU 2 ─ GPU 3
       ← TP group →

Node 1
GPU 4 ─ GPU 5 ─ GPU 6 ─ GPU 7
       ← TP group →


while using inter-node communication for parallelism patterns where the communication overhead is more manageable.

19. Can TP, PP and DP be combined?

Answer: Absolutely. Large-scale LLM training commonly combines them.

For example:

TP = 4
PP = 8
DP = 4

Total GPUs = 4 × 8 × 4 = 128


Conceptually:

TP=4: 4 GPUs cooperate on each layer.
PP=8: the model is divided into 8 pipeline stages.
DP=4: 4 copies of the overall distributed model process different data.

This allows the system to address three different scaling requirements simultaneously: computation, model size, and data/throughput.

20. Why are latency and throughput not simply inverses?

Answer:

Latency measures how long one unit of work takes, while throughput measures how much total work the system completes per unit time. A system can process many requests concurrently and therefore achieve high throughput while an individual request still experiences relatively high latency.

For LLMs:

Latency:
"How long does THIS request take?"

Throughput:
"How many requests/tokens can I process per second?"


This distinction is especially important because training primarily optimizes throughput, while interactive inference often prioritizes latency.

⭐ 5 lines worth memorizing for interviews
Tensor Parallelism → split computation within a layer → mainly GPU-to-GPU tensor communication.

Pipeline Parallelism → split layers across GPUs → activation/gradient communication between stages.

Data Parallelism → replicate the model → different data on each GPU → gradient synchronization.

Intra-node → GPUs communicating within one physical machine.

Inter-node → GPUs communicating across physical machines.


And the most important distinction:

TP answers "How do multiple GPUs cooperate on one layer?"
PP answers "How do we distribute different layers across GPUs?"
DP answers "How do we process more data/requests in parallel?"

# LLM Parallelism — Interview Questions & Answers

## 🟢 Easy — 1-liners

### 1. What is latency?

**Answer:** Latency is the time required to complete a single unit of work, such as one LLM request.

---

### 2. What is throughput?

**Answer:** Throughput is the amount of work completed per unit of time, such as requests/sec or tokens/sec.

---

### 3. What is Tensor Parallelism?

**Answer:** Tensor Parallelism splits tensors or matrix computations within a layer across multiple GPUs.

---

### 4. What is Pipeline Parallelism?

**Answer:** Pipeline Parallelism splits different layers of a model across different GPUs.

---

### 5. What is Data Parallelism?

**Answer:** Data Parallelism replicates the model across GPUs and gives each replica different data.

---

### 6. What is a node?

**Answer:** A node is a physical machine that can contain one or more GPUs.

---

### 7. What is intra-node communication?

**Answer:** Communication between GPUs located within the same physical machine.

---

### 8. What is inter-node communication?

**Answer:** Communication between GPUs located on different physical machines.

---

# 🟡 Medium — 2-liners

### 9. How does Tensor Parallelism improve performance?

**Answer:** It divides a large tensor/matrix operation across multiple GPUs so they perform portions of the computation simultaneously. The resulting partial tensors are then combined through GPU-to-GPU communication.

---

### 10. How does Data Parallelism improve throughput?

**Answer:** Each GPU maintains a model replica and processes a different batch of data simultaneously. Therefore, more samples/tokens can be processed per unit time.

---

### 11. What communication occurs in Pipeline Parallelism?

**Answer:** Pipeline stages exchange activation tensors during the forward pass and gradients during the backward pass. This is generally point-to-point communication between neighboring stages.

---

### 12. Why is Tensor Parallelism communication-heavy?

**Answer:** Multiple GPUs must frequently exchange partial results while computing the same layer. Therefore, high-bandwidth GPU interconnects are particularly important for TP.

---

### 13. Why doesn't Data Parallelism usually reduce single-request latency?

**Answer:** DP gives different data to different model replicas rather than splitting the computation of one request across GPUs. Its main purpose is increasing the number of requests/samples processed concurrently.

---

### 14. What is the pipeline bubble?

**Answer:** The pipeline bubble is the period when some pipeline stages are idle while the pipeline is being filled or drained. Using more microbatches helps reduce this idle time.

---

### 15. Can Tensor Parallelism be inter-node?

**Answer:** Yes, TP can span multiple nodes, but this introduces network communication and is generally more expensive than keeping the TP group within one node.

---

# 🔴 Tougher — Detailed Answers

### 16. Compare Tensor Parallelism and Pipeline Parallelism.

**Answer:**

Tensor Parallelism splits the computation within a layer across multiple GPUs:

```text
Layer
├── GPU 0 → portion of computation
└── GPU 1 → portion of computation

Pipeline Parallelism instead splits the layers themselves:

GPU 0 → Layers 1–8
GPU 1 → Layers 9–16
GPU 2 → Layers 17–24
```

Therefore, TP requires frequent communication between GPUs working on the same layer, while PP primarily communicates activations between pipeline stages.


### 17. How would you choose between TP, PP and DP?

**Answer:**

It depends on the bottleneck:

```
Model doesn't fit on one GPU
        ↓
      TP / PP
Need faster computation of a large layer
        ↓
        TP
Need higher training/serving throughput
        ↓
        DP
Very large model + many GPUs
        ↓
      TP + PP + DP
```
**Summary:**

- TP is useful for splitting computation within layers.
- PP distributes model layers across devices.
- DP creates multiple model replicas to process different data concurrently.

--- 
### 18. Why is TP usually kept within a node?

**Answer:**

- TP involves frequent synchronization and exchange of partial tensors.

- If those GPUs are on different nodes, communication must travel through the network, which generally has higher latency and lower effective bandwidth than high-speed intra-node GPU interconnects.

- Therefore, a common strategy is:

```
Node 0

GPU 0 ─ GPU 1 ─ GPU 2 ─ GPU 3
        ← TP group →
Node 1

GPU 4 ─ GPU 5 ─ GPU 6 ─ GPU 7
        ← TP group →
```

- while using inter-node communication for parallelism patterns where the communication overhead is more manageable.
---
### 19. Can TP, PP and DP be combined?

**Answer:**

- Absolutely. Large-scale LLM training commonly combines them.

**For example:**
```
TP = 4
PP = 8
DP = 4

Total GPUs = 4 × 8 × 4 = 128

Conceptually:

TP = 4
→ 4 GPUs cooperate on each layer.

PP = 8
→ The model is divided into 8 pipeline stages.

DP = 4
→ 4 copies of the overall distributed model process different data.
```

- This allows the system to address three different scaling requirements simultaneously:

Computation → TP
Model size → PP
Data / throughput → DP

---
### 20. Why are latency and throughput not simply inverses?

**Answer:**

- Latency measures how long one unit of work takes, while throughput measures how much total work the system completes per unit time.

- A system can process many requests concurrently and therefore achieve high throughput while an individual request still experiences relatively high latency.

**For LLMs:**

**Latency:**

"How long does THIS request take?"

**Throughput:**

"How many requests/tokens can I process per second?"

This distinction is especially important because training primarily optimizes throughput, while interactive inference often prioritizes latency.